<a href="https://colab.research.google.com/github/cbonnin88/Python-SQL-For-Product/blob/main/Music_Now.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import plotly.express as px
import random
from datetime import datetime, timedelta

# **1. Generate the mock dataset**

In [2]:
# Keeping the random data the same time every time we run it
random.seed(42)

In [3]:
base_date = datetime(2023, 10, 1)
event_types = ['song_play', 'song_play', 'song_play', 'login', 'ad_click'] # weighted to have more song plays

In [10]:
product_data = {
    "user_id": [random.randint(100, 150) for _ in range(1000)],
    # Generate random dates within a 7-day window
    "event_date": [(base_date + timedelta(days=random.randint(0, 6))).strftime('%Y-%m-%d') for _ in range(1000)],
    "event_type": [random.choice(event_types) for _ in range(1000)],
    # Random duration between 30 and 300 seconds. Some will be None (missing data!)
    "duration_seconds": [random.randint(30, 300) if random.random() > 0.1 else None for _ in range(1000)]
}

df_events = pl.DataFrame(product_data).with_columns(
    pl.col("event_date").str.strptime(pl.Date, "%Y-%m-%d")
)

In [29]:
df_events = df_events.with_columns(
    # Generate a temporary Series of random integers, one for each row.
    pl.Series([random.randint(30, 300) for _ in range(df_events.height)], dtype=pl.Int64).alias("random_durations_temp")
).with_columns(
    pl.when(pl.col("event_type") == "song_play")
    .then(pl.col("random_durations_temp"))
    .otherwise(None)
    .alias("duration_seconds")
).drop("random_durations_temp") # Drop the temporary column after use

In [16]:
df_music = pl.DataFrame(product_data)

In [17]:
print('Raw Data Preview:')
display(df_music.head())

Raw Data Preview:


user_id,event_date,event_type,duration_seconds
i64,str,str,i64
131,"""2023-10-01""","""song_play""",30
129,"""2023-10-06""","""login""",null
104,"""2023-10-06""","""song_play""",175
105,"""2023-10-03""","""song_play""",287
135,"""2023-10-06""","""song_play""",109


# **Data Cleaning with Polars**

In [18]:
df_music_cleaned = (
    df_music
    .with_columns(
        pl.col('event_date').str.strptime(pl.Date,'%Y-%m-%d')
    )
    .drop_nulls(subset=['duration_seconds'])
    .filter(pl.col('event_type') == 'song_play')
)

In [19]:
print('\nCleaned Data Preview:')
display(df_music_cleaned.head())


Cleaned Data Preview:


user_id,event_date,event_type,duration_seconds
i64,date,str,i64
131,2023-10-01,"""song_play""",30
104,2023-10-06,"""song_play""",175
105,2023-10-03,"""song_play""",287
135,2023-10-06,"""song_play""",109
142,2023-10-06,"""song_play""",168


# **Calculating Product Metrics**

- DAU (Daily Active Users)
- Total Streams
- Average Listen Time per day.


# **Metric 1: Daily Active Users (DAU)**

- Calculate the total number of unique users interacting with the app daily

In [20]:
df_dau = (
    df_events
    .group_by('event_date')
    .agg(pl.col('user_id').n_unique().alias('dau'))
    .sort('event_date')
)

display(df_dau.head())

event_date,dau
date,u32
2023-10-01,50
2023-10-02,48
2023-10-03,49
2023-10-04,48
2023-10-05,45


In [21]:
fig_dau = px.line(
    df_dau,
    x='event_date',
    y='dau',
    title='Metric 1: Daily Active Users (DAU)',
    labels={'event_date':'Date','dau':'Unique Active Users'},
    markers=True
)

fig_dau.update_layout(template='plotly_white',yaxis_rangemode='tozero')
fig_dau.show()

# **Metric 2: Total Daily Streams**

- How many streams happened each day ?

In [22]:
# Filter for song plays, group by date, and count the number of rows
df_streams = (
    df_events
    .filter(pl.col('event_type') == 'song_play')
    .group_by('event_date')
    .agg(pl.len().alias('total_streams'))
    .sort('event_date')
)

display(df_streams.head())

event_date,total_streams
date,u32
2023-10-01,86
2023-10-02,99
2023-10-03,95
2023-10-04,77
2023-10-05,77


In [24]:
fig_streams = px.bar(
    df_streams,
    x='event_date',
    y='total_streams',
    title='Metric 2: Total Daily Streams',
    color='event_date',
    labels={'event_date':'Date','total_streams':'Total Songs Played'},
    text_auto=True,
    color_discrete_sequence = px.colors.sequential.Viridis_r
)

fig_streams.update_layout(template='plotly_white')
fig_streams.show()

# **Metric 3: Average Listen Time per Day**

- The average length of those song plays per day.

In [30]:
df_listen_time = (
    df_events
    .filter(pl.col('event_type') == 'song_play')
    .drop_nulls(subset=['duration_seconds'])
    .group_by('event_date')
    .agg(pl.col('duration_seconds').mean().round(2).alias('avg_listen_time_sec'))
    .sort('event_date')
)

display(df_listen_time.head())

event_date,avg_listen_time_sec
date,f64
2023-10-01,177.35
2023-10-02,173.14
2023-10-03,155.86
2023-10-04,150.83
2023-10-05,155.77


In [34]:
fig_listen_time = px.line(
    df_listen_time,
    x='event_date',
    y='avg_listen_time_sec',
    title='Metric 3: Average Listen Time per Day (Seconds)',
    labels={'event_date':'Date','avg_listen_time_sec':'Avg Listen Time (s)'},
    markers=True
)

fig_listen_time.update_layout(template='plotly_white',yaxis_rangemode='tozero')
fig_listen_time.show()

# **Overall Conversion Rate**

In [42]:
# Counting total unique users in the app

total_users = df_events.select('user_id').n_unique()

In [43]:
# Filter for only 'upgrade' events, then count unique users

upgraded_users = (
    df_events
    .filter(pl.col('event_type')== 'upgrade')
    .select('user_id')
    .n_unique()
)

In [44]:
# Calculate the percentage

conversion_rate = (upgraded_users / total_users) * 100

In [46]:
print(f'Total Users: {total_users}')
print(f'Upgraded Users: {upgraded_users}')
print(f'Overall Conversion Rate: {conversion_rate:.2f}%')

Total Users: 51
Upgraded Users: 0
Overall Conversion Rate: 0.00%


# **Calculate Day-1 Retention Rate (Cohort Analysis)**


- Finding a user's first day, finding how many days after that first day they were active, and aggregating those numbers.

In [47]:
# Step 1: Finding the first date each user was active (The Cohort date)

first_active_day = (
    df_events
    .group_by('user_id')
    .agg(pl.col('event_date').min().alias('cohort_date'))
)

In [48]:
# Step 2: Joins this back to the main events table so every row know its users cohort date

df_retention = df_events.join(first_active_day, on='user_id')

In [49]:
# Step 3: Calculate the difference in days between the event and the cohort date

df_retention = df_retention.with_columns(
    (pl.col('event_date') - pl.col('cohort_date')).dt.total_days().alias('days_since_first_active')
)

In [54]:
# Step 4: Calculate the size of each cohort (how many people started on each day)

cohort_sizes = (
    df_retention
    .group_by('cohort_date')
    .agg(pl.col('user_id').n_unique().alias('cohort_size'))
)

In [55]:
# Step E: Count how many unique users came back exactly 1 day later

day_1_retained = (
    df_retention
    .filter(pl.col('days_since_first_active') == 1)
    .group_by('cohort_date')
    .agg(pl.col('user_id').n_unique().alias('day_1_active_users'))
)

In [56]:
# Step 6: Merge cohort sizes and Day-1 active users to calculate the final percentage

retention_metrics = (
    cohort_sizes
    .join(day_1_retained, on='cohort_date', how='left')
    .fill_null(0)
    .with_columns(
        (pl.col('day_1_active_users') / pl.col('cohort_size')* 100).round(2).alias(('day_1_retention_pct'))
    )
    .sort('cohort_date')
)

In [57]:
print('\nDay-1 Retention Metrics by Cohort:')
display(retention_metrics)


Day-1 Retention Metrics by Cohort:


cohort_date,cohort_size,day_1_active_users,day_1_retention_pct
date,u32,u32,f64
2023-10-01,50,47,94.0
2023-10-02,1,1,100.0


In [60]:
fig_retention = px.bar(
    retention_metrics,
    x='cohort_date',
    y='day_1_retention_pct',
    title='Day-1 Retention Rate by Cohort - Music Now',
    labels={
        'cohort_date': 'Cohort Date (First Active Day)',
        'day_1_retention_pct': 'Day-1 Retention (%)'
    },
    text_auto=True,
    color='cohort_date',
    color_discrete_sequence = px.colors.sequential.Viridis_r

)

fig_retention.update_layout(
    template='plotly_white',
    yaxis=dict(range=[0,100]),
    xaxis_tickangle=-45
)

fig_retention.show()